# 02 — Feature Engineering

Phân tích các features được tạo ra từ dữ liệu MovieLens.

**Nội dung:**
1. Load processed data
2. User features — phân phối, genre preferences
3. Item features — popularity, year, genre encoding, TF-IDF tags
4. Interaction matrix — sparsity visualization
5. Feature correlation heatmap

In [ ]:
import sys
from pathlib import Path

PROJECT_ROOT = Path("../").resolve()
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

import warnings

warnings.filterwarnings("ignore")
import logging

logging.disable(logging.CRITICAL)

import matplotlib.colors as mcolors
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

from src.config import settings
from src.features.interaction import build_interaction_matrix
from src.features.item_features import build_item_features
from src.features.user_features import build_user_features
from src.training.train import load_processed_data

plt.rcParams.update({"figure.dpi": 110, "axes.grid": True, "grid.alpha": 0.3})
print("Imports OK")

## 1. Load Processed Data

In [ ]:
data = load_processed_data(settings.data_processed_dir)
train_df = data["train"]
movies_df = data["movies"]

# Load tags if available
tags_path = settings.data_processed_dir / "tags.parquet"
tags_df = pd.read_parquet(tags_path) if tags_path.exists() else None

n_users = int(train_df["user_idx"].max()) + 1
n_items = int(train_df["movie_idx"].max()) + 1

print(f"Train rows: {len(train_df):,}")
print(f"Users     : {n_users:,}")
print(f"Items     : {n_items:,}")
print(f"Tags      : {len(tags_df):,}" if tags_df is not None else "Tags: not available")

## 2. User Features

In [ ]:
user_feat = build_user_features(train_df, movies_df)
print(f"User feature matrix: {user_feat.shape}")
user_feat.describe().round(3)

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(14, 4))
fig.suptitle("User Features Distribution", fontsize=13, fontweight="bold")

axes[0].hist(user_feat["avg_rating"], bins=50, color="#4e9dd4", edgecolor="white")
axes[0].set_xlabel("Average Rating")
axes[0].set_title("User Avg Rating")

axes[1].hist(user_feat["num_ratings"], bins=60, color="#fd8d3c", edgecolor="white", log=True)
axes[1].set_xlabel("Number of Ratings")
axes[1].set_ylabel("Users (log scale)")
axes[1].set_title("User Activity (log scale)")

axes[2].hist(user_feat["rating_std"], bins=50, color="#74c476", edgecolor="white")
axes[2].set_xlabel("Rating Std Dev")
axes[2].set_title("User Rating Consistency")

plt.tight_layout()
plt.savefig("feature_user_distributions.png", bbox_inches="tight")
plt.show()

In [ ]:
# Genre preference heatmap (sample 500 users)
genre_pref_cols = [c for c in user_feat.columns if c.startswith("genre_pref_")]

if genre_pref_cols:
    sample = user_feat[genre_pref_cols].sample(min(500, len(user_feat)), random_state=42)
    genre_means = user_feat[genre_pref_cols].mean().sort_values(ascending=False)
    genre_labels = [c.replace("genre_pref_", "") for c in genre_means.index]

    fig, ax = plt.subplots(figsize=(12, 4))
    ax.bar(genre_labels, genre_means.values, color="#9e9ac8", edgecolor="white")
    ax.set_xticklabels(genre_labels, rotation=45, ha="right")
    ax.set_ylabel("Mean Genre Preference")
    ax.set_title("Average User Genre Preferences", fontsize=13, fontweight="bold")
    plt.tight_layout()
    plt.savefig("feature_user_genre_pref.png", bbox_inches="tight")
    plt.show()
else:
    print("No genre preference columns found")

## 3. Item Features

In [ ]:
item_feat = build_item_features(train_df, movies_df, tags=tags_df)
print(f"Item feature matrix: {item_feat.shape}")

# Column groups
numeric_cols = ["avg_rating", "num_ratings", "popularity_score", "year"]
genre_cols = [
    c for c in item_feat.columns if c not in numeric_cols and not c.startswith("tag_tfidf_")
]
tfidf_cols = [c for c in item_feat.columns if c.startswith("tag_tfidf_")]

print(f"Numeric features : {len(numeric_cols)}")
print(f"Genre features   : {len(genre_cols)}")
print(f"TF-IDF features  : {len(tfidf_cols)}")
item_feat[numeric_cols].describe().round(3)

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(14, 4))
fig.suptitle("Item Features Distribution", fontsize=13, fontweight="bold")

axes[0].hist(item_feat["avg_rating"].dropna(), bins=50, color="#4e9dd4", edgecolor="white")
axes[0].set_xlabel("Average Rating")
axes[0].set_title("Item Avg Rating")

axes[1].hist(item_feat["popularity_score"].dropna(), bins=60, color="#fd8d3c", edgecolor="white")
axes[1].set_xlabel("log1p(num_ratings)")
axes[1].set_title("Item Popularity Score")

year_vals = item_feat["year"].dropna()
axes[2].hist(year_vals, bins=40, color="#74c476", edgecolor="white", range=(1900, 2025))
axes[2].set_xlabel("Release Year")
axes[2].set_title("Movie Release Year")

plt.tight_layout()
plt.savefig("feature_item_distributions.png", bbox_inches="tight")
plt.show()

In [ ]:
# Genre distribution across items
if genre_cols:
    genre_totals = item_feat[genre_cols].sum().sort_values(ascending=False)
    fig, ax = plt.subplots(figsize=(11, 4))
    ax.bar(genre_totals.index, genre_totals.values, color="#9e9ac8", edgecolor="white")
    ax.set_xticklabels(genre_totals.index, rotation=45, ha="right")
    ax.set_ylabel("Number of Movies")
    ax.set_title("Movies per Genre (multi-hot encoding)", fontsize=13, fontweight="bold")
    plt.tight_layout()
    plt.savefig("feature_item_genre_dist.png", bbox_inches="tight")
    plt.show()

## 4. Interaction Matrix Visualization

In [ ]:
explicit_matrix, implicit_matrix = build_interaction_matrix(
    train_df,
    n_users,
    n_items,
    implicit_threshold=settings.implicit_threshold,
)

sparsity = 1 - implicit_matrix.nnz / (n_users * n_items)
print(f"Interaction matrix: {implicit_matrix.shape}")
print(f"Non-zeros (nnz)   : {implicit_matrix.nnz:,}")
print(f"Sparsity          : {sparsity:.4%}")
print(f"Density           : {1 - sparsity:.4%}")

In [ ]:
# Visualize a 200x200 submatrix of the interaction matrix
SAMPLE = 200
sub = implicit_matrix[:SAMPLE, :SAMPLE].toarray()

fig, ax = plt.subplots(figsize=(7, 6))
im = ax.imshow(sub, cmap="Blues", aspect="auto", interpolation="nearest")
ax.set_title(
    f"Interaction Matrix (first {SAMPLE}×{SAMPLE} submatrix)\nSparsity: {sparsity:.2%}",
    fontsize=12,
    fontweight="bold",
)
ax.set_xlabel("Item index")
ax.set_ylabel("User index")
plt.colorbar(im, ax=ax, label="Implicit feedback (0/1)")
plt.tight_layout()
plt.savefig("feature_interaction_matrix.png", bbox_inches="tight")
plt.show()

In [ ]:
# Row/column nnz distributions
row_nnz = np.diff(implicit_matrix.indptr)  # nnz per user
col_nnz = np.bincount(implicit_matrix.indices, minlength=n_items)  # nnz per item

fig, axes = plt.subplots(1, 2, figsize=(13, 4))
fig.suptitle("Interaction Matrix — NNZ Distribution", fontsize=13, fontweight="bold")

axes[0].hist(row_nnz, bins=60, color="#4e9dd4", edgecolor="white", log=True)
axes[0].set_xlabel("Interactions per User")
axes[0].set_ylabel("Count (log scale)")
axes[0].set_title(f"User Activity  (mean={row_nnz.mean():.1f})")

axes[1].hist(col_nnz[col_nnz > 0], bins=60, color="#74c476", edgecolor="white", log=True)
axes[1].set_xlabel("Interactions per Item")
axes[1].set_ylabel("Count (log scale)")
axes[1].set_title(f"Item Popularity (mean={col_nnz[col_nnz > 0].mean():.1f})")

plt.tight_layout()
plt.savefig("feature_nnz_distributions.png", bbox_inches="tight")
plt.show()

## 5. Feature Correlation

In [ ]:
# Correlation among numeric item features + genre proportions
corr_cols = numeric_cols + genre_cols[:10]  # top-10 genres for readability
corr_data = item_feat[corr_cols].dropna()
corr_matrix = corr_data.corr()

short_labels = [c.replace("genre_pref_", "") for c in corr_cols]

fig, ax = plt.subplots(figsize=(10, 8))
im = ax.imshow(corr_matrix.values, cmap="RdBu_r", vmin=-1, vmax=1, aspect="auto")
ax.set_xticks(range(len(short_labels)))
ax.set_yticks(range(len(short_labels)))
ax.set_xticklabels(short_labels, rotation=45, ha="right", fontsize=9)
ax.set_yticklabels(short_labels, fontsize=9)
ax.set_title("Item Feature Correlation Matrix", fontsize=13, fontweight="bold")
plt.colorbar(im, ax=ax)
for i in range(len(corr_cols)):
    for j in range(len(corr_cols)):
        val = corr_matrix.values[i, j]
        ax.text(
            j,
            i,
            f"{val:.2f}",
            ha="center",
            va="center",
            fontsize=7,
            color="white" if abs(val) > 0.5 else "black",
        )
plt.tight_layout()
plt.savefig("feature_correlation_matrix.png", bbox_inches="tight")
plt.show()

In [ ]:
# Popularity vs average rating scatter
fig, ax = plt.subplots(figsize=(8, 5))
sc = ax.scatter(
    item_feat["popularity_score"],
    item_feat["avg_rating"],
    alpha=0.3,
    s=5,
    c=item_feat["num_ratings"],
    cmap="YlOrRd",
    norm=mcolors.LogNorm(),
)
plt.colorbar(sc, ax=ax, label="num_ratings")
ax.set_xlabel("Popularity Score (log1p)")
ax.set_ylabel("Average Rating")
ax.set_title("Item Popularity vs Average Rating", fontsize=13, fontweight="bold")
plt.tight_layout()
plt.savefig("feature_popularity_vs_rating.png", bbox_inches="tight")
plt.show()

corr = item_feat[["popularity_score", "avg_rating"]].corr().iloc[0, 1]
print(f"Pearson correlation (popularity vs avg_rating): {corr:.4f}")

## Summary

In [ ]:
print("=" * 55)
print("  Feature Engineering Summary")
print("=" * 55)
print(f"  Users               : {n_users:>10,}")
print(f"  Items               : {n_items:>10,}")
print(f"  Interaction nnz     : {implicit_matrix.nnz:>10,}")
print(f"  Matrix sparsity     : {sparsity:>10.4%}")
print("-" * 55)
print(f"  User feature dim    : {user_feat.shape[1]:>10}")
print(f"  Item feature dim    : {item_feat.shape[1]:>10}")
print(f"    ↳ numeric         : {len(numeric_cols):>10}")
print(f"    ↳ genre (multi-hot): {len(genre_cols):>9}")
print(f"    ↳ tag TF-IDF      : {len(tfidf_cols):>10}")
print("=" * 55)